In [1]:
import functional_functions as ff
from nilearn.maskers import SurfaceLabelsMasker
from nilearn.connectome import ConnectivityMeasure
from nilearn.surface import SurfaceImage
import numpy as np
import pandas as pd
from pathlib import Path

/srv/conda/envs/notebook/lib/python3.10/site-packages/google/api_core/_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
pixdim[1,2,3] should be non-zero; setting 0 dims to 1


In [2]:
# create mesh and parcellate based on MMP
labels_image = ff.create_mmp_mesh()

In [3]:
# create look up table with labels 
lut = ff.create_mmp_lookup()

In [4]:
# make and fit the masker to the ROIs from Glasser 
surface_lbls_masker = SurfaceLabelsMasker(labels_img=labels_image, lut=lut).fit()

In [5]:
# read in participant csv 
df = pd.read_csv('test_split.csv')
subjects = df['Subject'].values

In [7]:
np.shape(subjects)

(174,)

In [43]:
# instantiate storage matrix 
mat = []

In [ ]:
# loop through all participants 
for sub_num in subjects:
    
    (all_left_hemi, all_right_hemi) = ff.ciis_to_giis(sub_num)
    
    # put our data in SurfaceImage object 
    func_data = SurfaceImage(
        mesh=labels_image.mesh,
        data= {
            "left": all_left_hemi.T,
            "right": all_right_hemi.T
        }
    )
    roi_time_series = surface_lbls_masker.transform(func_data)
    
    # make correlation matrix 
    correlation_measure = ConnectivityMeasure(kind='correlation', vectorize=True, discard_diagonal=True)
    correlation_matrix = correlation_measure.fit_transform([roi_time_series])[0]

    # add subject number to beginning of vectorized corrmat
    # add to matrix
    mat.append(np.insert(correlation_matrix, 0, sub_num))
    print(sub_num)
    

In [45]:
np.shape(mat)

(692, 64981)

In [46]:
# save to home directory
home = Path.home()
np.save(home/'training_corrmats_all.npy', mat)